In [2]:
from rerank_reward import *

import pandas as pd


file = '/mnt/local2/wxy/herb_rec+RLv2/data/tcm_herb_rerank_v2.2_no_dup_c50/test.parquet'
file = '/mnt/local2/wxy/herb_rec+RLv2/data/tcm_herb_rerank_c30_no_dup_filter_v10/test.parquet'
file = '/mnt/local2/wxy/herb_rec+RLv2/data/tcm_herb_rerank_c30_full_v10/test.parquet'
file = '/mnt/local2/wxy/herb_rec+RLv2/data/tcm_herb_rerank_c30_no_dup_filter_v10/train.parquet'

data = pd.read_parquet(file)

In [15]:
gt = None

for reward, symp in zip(data['reward_model'], data['symptoms']):
    if symp == '气喘 浮肿':
        print(reward)
        print(symp)
        ground_truth = reward['ground_truth']

{'ground_truth': {'candidate_herbs': array(['甘草', '茯苓', '桑', '槟榔', '杏', '枳壳', '木香', '防己', '白术', '半夏', '杏仁',
       '当归', '大腹皮', '陈皮', '大黄', '人参', '黄芩', '木通', '麻黄', '桑白皮', '防风', '郁李',
       '干姜', '生姜', '郁李仁', '桔梗', '甘遂', '芍药', '厚朴', '五味子'], dtype=object), 'gt_herbs': array(['生姜', '桔梗'], dtype=object), 'gt_herbs_list': array([array(['大腹皮', '大黄', '桑', '槟榔', '甘草', '羌活', '防己'], dtype=object),
       array(['甘遂'], dtype=object),
       array(['吴茱萸', '木瓜', '木香', '柴胡', '槟榔', '沉香', '芍药', '茯苓', '赤芍', '高良姜'],
             dtype=object)                                                  ,
       array(['商陆', '大戟', '巴豆', '桑', '桑白皮', '椒目', '槟榔', '泽泻', '甘遂', '芫花', '葶苈子',
              '雄黄'], dtype=object)                                              ,
       array(['杏', '杏仁'], dtype=object),
       array(['杏', '杏仁', '皂荚', '郁李', '郁李仁', '防己', '马兜铃', '鸡子', '鸡子黄'],
             dtype=object)                                            ,
       array(['大黄', '杏', '杏仁', '泽泻', '甘遂', '白术', '细辛', '芫花', '茯苓', '防己

In [16]:
from rerank_reward import _get_reward_profile, _safe_list, _collect_gt_variants, get_theoretical_optimal_score

profile = _get_reward_profile()

# ground_truth = data['reward_model'][2]['ground_truth']

candidate_herbs = _safe_list(ground_truth.get("candidate_herbs", []))
candidate_set   = set(candidate_herbs)
gt_variants     = _collect_gt_variants(ground_truth, candidate_set)

best_score, target_gt_variant = get_theoretical_optimal_score(gt_variants, candidate_herbs, profile["metric_weights"])

In [7]:
best_score

0.82

In [17]:
target_gt_variant

['杏', '杏仁', '郁李', '郁李仁', '防己']

In [18]:

pred_raw = [
        "桑",
        "甘草",
        "茯苓",
        "槟榔",
        "杏",
        "枳壳",
        "木香",
        "防己",
        "白术",
        "半夏"
    ]

copy_num_min = min(len(pred_raw), len(target_gt_variant))
base_top = candidate_herbs[:copy_num_min] 
pred_top = pred_raw[:copy_num_min]
copy_count = sum(1 for p, b in zip(pred_top, base_top) if p == b)
similarity = copy_count / copy_num_min if copy_num_min > 0 else 0.0
copy_penalty = 2 * max(0, similarity - 0.5)

In [21]:
print(base_top)
print(pred_top)
print(copy_count)
print(similarity)
print(copy_penalty)

['甘草', '茯苓', '桑', '槟榔', '杏']
['桑', '甘草', '茯苓', '槟榔', '杏']
2
0.4
0


In [23]:
def pairwise_win_reward(ranked_drugs: list, gt_drugs: list | set) -> float:
    gt_set = set(gt_drugs)
    drug_pos = {d: i for i, d in enumerate(ranked_drugs)}

    total_wins = 0
    total_pairs = 0

    for gt in gt_set:
        if gt not in drug_pos:
            continue
        g_pos = drug_pos[gt]
        # 所有排在 GT 前面的药物
        for other in ranked_drugs[:g_pos]:
            total_pairs += 1
            if other not in gt_set:
                total_wins += 1

    if total_pairs == 0:
        return 0.0
    win_rate = total_wins / total_pairs
    return win_rate ** 0.7

def compute_normalized_srr(ranked_drugs: list, gt_drugs: list, normalize: bool = True) -> float:
    """
    计算多标签倒数和 (Sum of Reciprocal Ranks, SRR)
    
    参数:
        ranked_drugs: 模型输出的药物排序列表 (预测)
        gt_drugs: 真实的药物列表 (Ground Truth)
        normalize: 是否归一化到 0~1 之间 (强烈建议强化学习时设为 True)
        
    返回:
        float: SRR 得分
    """
    if not gt_drugs or not ranked_drugs:
        return 0.0

    gt_set = set(gt_drugs)
    srr_score = 0.0
    seen = set()
    
    # 记录当前有效的排名（跳过重复输出的废药）
    valid_rank = 1 

    for drug in ranked_drugs:
        if drug in seen:
            continue
        seen.add(drug)
        
        # 如果模型预测的药在 GT 中，累加 1/rank
        if drug in gt_set:
            srr_score += 1.0 / valid_rank
            
        valid_rank += 1

    if not normalize:
        return srr_score

    # 计算理想情况下的最高分 (Ideal SRR)
    # 完美的排序应该是把所有的 gt_drugs 排在最前面，即 rank = 1, 2, ..., len(gt)
    ideal_srr = sum(1.0 / r for r in range(1, len(gt_set) + 1))
    
    return srr_score / ideal_srr if ideal_srr > 0 else 0.0

def calc_f1(pred: list, gt: list, k=10) -> float:
    gt_set = set(gt)
    pred_top_k = pred[:k]
    pred_top_k_set = set(pred_top_k)
    true_positives = len(pred_top_k_set & gt_set)
    precision = true_positives / len(pred_top_k) if pred_top_k else 0.0
    recall = true_positives / len(gt_set) if gt_set else 0.0
    if precision + recall == 0:
        return 0.0
    f1_score = 2 * (precision * recall) / (precision + recall)
    return f1_score

def compute_pairwise_reward(ranked_drugs: list, candidate_herbs: list, gt_drugs: list) -> float:
    """
    计算 Pairwise 胜负奖励 (相当于 AUC)
    每当一个正确药物 (GT) 排在一个错误药物前面，就计作一次 Win。
    """
    if not gt_drugs or not candidate_herbs:
        return 0.0

    gt_set = set(gt_drugs)
    cand_set = set(candidate_herbs)
    
    # 区分出“该排在前面的药”和“该排在后面的药”
    relevant_drugs = [d for d in gt_set if d in cand_set]
    irrelevant_drugs = [d for d in cand_set if d not in gt_set]
    
    if not relevant_drugs or not irrelevant_drugs:
        return 0.0  # 全是对的或全是错的，无法构成 Pair
        
    # 构建当前预测的排名映射
    # 如果模型“漏掉”了某个候选药，我们惩罚它，给它一个无穷大的排名 (排在最最最下面)
    penalty_rank = len(candidate_herbs) + 100
    
    # 用字典记录位置：为了防作弊，使用去重且保留顺序的列表
    seen = set()
    dedup_ranked = []
    for d in ranked_drugs:
        if d not in seen:
            seen.add(d)
            dedup_ranked.append(d)
            
    rank_dict = {drug: idx for idx, drug in enumerate(dedup_ranked)}
    
    # 统计胜负
    wins = 0
    total_pairs = len(relevant_drugs) * len(irrelevant_drugs)
    
    for rel in relevant_drugs:
        rank_rel = rank_dict.get(rel, penalty_rank) # 正确药物的排名
        for irr in irrelevant_drugs:
            rank_irr = rank_dict.get(irr, penalty_rank) # 错误药物的排名
            
            # 如果正确药物排在了错误药物前面，得 1 分
            if rank_rel < rank_irr:
                wins += 1
                
    # 归一化到 0 ~ 1 之间
    return wins / total_pairs

In [48]:
candidate_herbs = [15, 41, 12, 27, 18, 71, 66, 295, 80, 45, 38, 26, 23, 102, 85, 16, 100, 116, 24, 366, 8, 37, 112, 29, 68, 89, 138, 347, 46, 178, 6, 189, 160, 17, 147, 203, 161, 54, 10, 32, 136, 101, 0, 86, 120, 166, 57, 386, 78, 9]
gt = [46, 9]

pw_pos = 0
srr_pos = 0
f1_pos = 0

try_num = 500

for _ in range(try_num):
    import random
    rerank = random.sample(candidate_herbs, 20)

    # print(candidate_herbs)
    # print(rerank)

    base_pw = compute_pairwise_reward(candidate_herbs, candidate_herbs, gt)
    base_srr  =compute_normalized_srr(candidate_herbs, gt)
    base_f1 = calc_f1(candidate_herbs, gt, k=5)


    rerank_pw = compute_pairwise_reward(rerank, candidate_herbs, gt)
    rerank_srr = compute_normalized_srr(rerank, gt)
    rerank_f1 = calc_f1(rerank, gt, k=5)
    
    if rerank_pw - base_pw > 0:
        pw_pos += 1
    if rerank_srr - base_srr > 0:
        srr_pos += 1
    if rerank_f1 - base_f1 > 0:
        f1_pos += 1
        
        
print(f"Pairwise: {pw_pos}")
print(f"SRR: {srr_pos}")
print(f"F1@5: {f1_pos}")


Pairwise: 316
SRR: 292
F1@5: 88
